# 🌊 UNIFIED MAMBA EDGE MASTER SCREENING & PROGRESSION BENCHMARK
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doantrongthai/flood_mamba_edge/blob/main/master_mamba_screening_and_progression.ipynb)

### 15 Architectures | Seed 42 | Size 128x128 | Dual Dataset (FloodKaggle & AIFloodSense)

This notebook executes an end-to-end evaluation covering:
1. **LCMamNet Systematic Bottleneck Resolution Progression (Steps 0 → 4)**:
   - **Step 0**: `lcmamnet` (Baseline, 1.15M params, 5440 tokens)
   - **Step 1**: `lcmam_step1` (Resolves Bottleneck 1: Decoupled Multi-Scale Tokens, 1344 tokens, -75% tokens)
   - **Step 2**: `lcmam_step2` (Resolves Bottlenecks 1 + 2: + Asymmetric Strip Depthwise ASD-Conv)
   - **Step 3**: `lcmam_step3` (Resolves Bottlenecks 1 + 2 + 3: + Spatial-Gated Dilated Refinement SGR)
   - **Step 4**: `aquamamnet` (Resolves Bottlenecks 1 + 2 + 3 + 4 simultaneously: + Lite-Boundary PixelShuffle Decoder + Mamba `expand=1`)
2. **10 Novel Flood-Domain Edge Mamba Architectures**:
   - `flowmamba`, `wavemamba`, `morphomamba`, `ripplemamba`, `aeromamba`, `topomamba`, `hydromamba`, `deltamamba`, `boundarymamba`, `resmambanet`

All models run sequentially with **Seed 42** on **both datasets** at resolution **128x128**. Results are compiled into automated Leaderboard and Component Progression tables.

In [ ]:
# 1. Environment Setup & Repository Initialization
import os, sys, shutil

REPO_URL = "https://github.com/doantrongthai/flood_mamba_edge.git"
REPO_DIR = "/content/flood_mamba_edge"

if "google.colab" in str(get_ipython()):
    print("Detected Google Colab environment.")
    if not os.path.exists(REPO_DIR):
        print(f"Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
else:
    print(f"Running locally / offline. Current directory: {os.getcwd()}")
    if os.path.exists("models"):
        sys.path.insert(0, os.getcwd())

print(f"Working Directory: {os.getcwd()}")
!python --version

In [ ]:
# 2. Install Dependencies & Hardware Acceleration
!pip install -q albumentations opencv-python-headless fvcore thop pandas tabulate colorama matplotlib

# Install official precompiled Mamba & Causal-Conv1d wheels on Colab (CUDA 12.x / PyTorch 2.x)
try:
    import mamba_ssm
    print("✓ mamba_ssm already installed.")
except ImportError:
    print("Installing official mamba-ssm and causal-conv1d...")
    !pip install -q causal-conv1d>=1.4.0 --no-build-isolation
    !pip install -q mamba-ssm>=2.2.2 --no-build-isolation

# Verification
import torch
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 3. Download & Prepare Datasets (FloodKaggle & AIFloodSense)
import subprocess

for ds in ["floodkaggle", "aifloodsense"]:
    print(f"Ensuring dataset '{ds}' is downloaded...")
    subprocess.run([sys.executable, "main.py", "--dataset", ds, "--download", "--epochs", "1", "--batch_size", "2", "--model", "unet", "--size", "128"], check=False)

print("✓ Datasets ready in data/ directory.")

In [ ]:
# 4. Master Architecture Registry & Experiment Configuration
SEED = 42
IMAGE_SIZE = 128
NUM_CLASSES = 1
EPOCHS = 15          # Fast screening epochs (use 50-100 for final training)
BATCH_SIZE = 16
LR = 1e-3
DATASETS = ["floodkaggle", "aifloodsense"]

# Group 1: Systematic Bottleneck Resolution Progression (Steps 0 -> 4)
PROGRESSION_MODELS = [
    "lcmamnet",     # Step 0: Baseline (1.15M params, 5440 tokens)
    "lcmam_step1",  # Step 1: 1 Bottleneck Resolved (Decoupled Multi-Scale Tokens, 1344 tokens)
    "lcmam_step2",  # Step 2: 2 Bottlenecks Resolved (+ Asymmetric Strip Depthwise ASD-Conv)
    "lcmam_step3",  # Step 3: 3 Bottlenecks Resolved (+ Spatial-Gated Dilated Refinement SGR)
    "aquamamnet",   # Step 4: All 4 Bottlenecks Resolved (+ Lite PixelShuffle Decoder + expand=1)
]

# Group 2: 10 Novel Flood-Domain Architectures
NOVEL_FLOOD_MODELS = [
    "flowmamba",      # Directional Strip Convolutions (1x7 & 7x1)
    "wavemamba",      # 2D Wavelet Haar Decomposition (High/Low frequency split)
    "morphomamba",    # Differentiable Morphological Gradient Gating (Dilation - Erosion)
    "ripplemamba",    # Concentric Multi-Stride Radial Scanning
    "aeromamba",      # Ultra-light UAV Drone Mamba (<0.1M params)
    "topomamba",      # Topographic Sobel Gradient-Modulated Mamba
    "hydromamba",     # Dual-Path Local CNN + Global Mamba Cross-Gating
    "deltamamba",     # Dilated Multi-Receptive Depthwise Convolutions (d=1,2,4)
    "boundarymamba",  # Internal High-Frequency Boundary Enhancement Prior
    "resmambanet",    # Inverted Residual Bottleneck with Mamba Token Mixer
]

# All 15 Models to Evaluate
ALL_MODELS = PROGRESSION_MODELS + NOVEL_FLOOD_MODELS

print(f"Total architectures registered: {len(ALL_MODELS)}")
print(f"Progression Study: {len(PROGRESSION_MODELS)} models")
print(f"Novel Architectures: {len(NOVEL_FLOOD_MODELS)} models")

In [ ]:
# 5. Automated Hardware & Complexity Profiling Table
import pandas as pd
import torch
from models import get_model
from utils.metrics import calculate_model_complexity, measure_inference_time

profiling_records = []
print("=" * 85)
print(f"{'ARCH':<16} | {'CATEGORY':<14} | {'PARAMS':<10} | {'GFLOPS':<8} | {'LATENCY':<10} | {'FPS':<8}")
print("=" * 85)

for model_name in ALL_MODELS:
    try:
        model = get_model(model_name, num_classes=NUM_CLASSES, input_size=IMAGE_SIZE, seed=SEED)
        complexity = calculate_model_complexity(model, input_size=IMAGE_SIZE)
        perf = measure_inference_time(model, input_size=IMAGE_SIZE, num_iterations=100)
        
        category = "Progression" if model_name in PROGRESSION_MODELS else "Novel Flood"
        profiling_records.append({
            "Architecture": model_name,
            "Category": category,
            "Params": complexity["params"],
            "GFLOPs": complexity["gflops"],
            "Latency (ms)": perf["latency_mean_ms"],
            "FPS": perf["fps"]
        })
        print(f"{model_name:<16} | {category:<14} | {complexity['params']:>10,d} | {complexity['gflops']:>8.3f} | {perf['latency_mean_ms']:>8.2f} ms | {perf['fps']:>8.1f}")
    except Exception as e:
        print(f"{model_name:<16} | FAILED PROFILING: {e}")

df_profile = pd.DataFrame(profiling_records)
df_profile.to_csv("hardware_profiling_results.csv", index=False)
print("=" * 85)
print("✓ Hardware profiling completed and saved to hardware_profiling_results.csv")

In [ ]:
# 6. UNIFIED SCREENING EXECUTION: Run all models on FloodKaggle & AIFloodSense (Seed 42)
import re, subprocess

screening_results = {}

for idx, model_name in enumerate(ALL_MODELS, 1):
    category = "Progression" if model_name in PROGRESSION_MODELS else "Novel Flood"
    print(f"\n{'#'*85}")
    print(f"[{idx}/{len(ALL_MODELS)}] BENCHMARKING: {model_name.upper()} ({category})")
    print(f"{'#'*85}")
    screening_results[model_name] = {}
    
    for ds in DATASETS:
        print(f"\n>>> Training {model_name} on {ds.upper()} (Seed {SEED}) <<<")
        cmd = [
            sys.executable, "main.py",
            "--dataset", ds,
            "--model", model_name,
            "--size", str(IMAGE_SIZE),
            "--epochs", str(EPOCHS),
            "--batch_size", str(BATCH_SIZE),
            "--lr", str(LR),
            "--seed", str(SEED)
        ]
        
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        full_log = []
        for line in proc.stdout:
            full_log.append(line)
            if "Epoch [" in line or "mIoU:" in line or "Best" in line or "EVALUATION" in line:
                print(line, end="")
        proc.wait()
        
        log_str = "".join(full_log)
        
        # Parse metrics using regex
        miou = None
        dice = None
        loss = None
        
        miou_match = re.findall(r"mIoU[:=]\s*([0-9.]+)", log_str)
        if miou_match:
            miou = float(miou_match[-1])
            
        dice_match = re.findall(r"(?:Dice|F1)[:=]\s*([0-9.]+)", log_str)
        if dice_match:
            dice = float(dice_match[-1])
            
        loss_match = re.findall(r"(?:Val Loss|Loss)[:=]\s*([0-9.]+)", log_str)
        if loss_match:
            loss = float(loss_match[-1])
            
        screening_results[model_name][ds] = {
            "mIoU": miou,
            "Dice": dice,
            "Val_Loss": loss
        }
        print(f"--> [{model_name} on {ds}] Completed: mIoU={miou}, Dice={dice}, Val_Loss={loss}")

print("\n✓ All training runs finished successfully!")

In [ ]:
# 7. SUMMARY LEADERBOARD & PROGRESSION STUDY TABLES
import pandas as pd
from tabulate import tabulate

master_rows = []
for m in ALL_MODELS:
    fk = screening_results.get(m, {}).get("floodkaggle", {})
    afs = screening_results.get(m, {}).get("aifloodsense", {})
    
    fk_miou = fk.get("mIoU") or 0.0
    fk_dice = fk.get("Dice") or 0.0
    afs_miou = afs.get("mIoU") or 0.0
    afs_dice = afs.get("Dice") or 0.0
    mean_miou = (fk_miou + afs_miou) / 2.0
    mean_dice = (fk_dice + afs_dice) / 2.0
    
    prof_row = df_profile[df_profile["Architecture"] == m]
    params = prof_row["Params"].values[0] if len(prof_row) > 0 else 0
    gflops = prof_row["GFLOPs"].values[0] if len(prof_row) > 0 else 0.0
    fps = prof_row["FPS"].values[0] if len(prof_row) > 0 else 0.0
    latency = prof_row["Latency (ms)"].values[0] if len(prof_row) > 0 else 0.0
    category = "Progression" if m in PROGRESSION_MODELS else "Novel Flood"
    
    master_rows.append({
        "Architecture": m,
        "Category": category,
        "Params": params,
        "GFLOPs": gflops,
        "FPS": fps,
        "Latency (ms)": latency,
        "FloodKaggle mIoU": fk_miou,
        "FloodKaggle Dice": fk_dice,
        "AIFloodSense mIoU": afs_miou,
        "AIFloodSense Dice": afs_dice,
        "Mean mIoU": mean_miou,
        "Mean Dice": mean_dice
    })

df_master = pd.DataFrame(master_rows)
df_master.to_csv("master_screening_results.csv", index=False)

# -------------------------------------------------------------
# TABLE 1: BOTTLENECK RESOLUTION PROGRESSION STUDY (Steps 0 -> 4)
# -------------------------------------------------------------
df_prog = df_master[df_master["Category"] == "Progression"].copy()
step_map = {
    "lcmamnet": "Step 0 (Baseline LCMamNet)",
    "lcmam_step1": "Step 1 (+ Decoupled Tokens)",
    "lcmam_step2": "Step 2 (+ ASD-Conv)",
    "lcmam_step3": "Step 3 (+ SGR)",
    "aquamamnet": "Step 4 (Proposed AquaMamNet: All 4 Resolved)"
}
df_prog["Step"] = df_prog["Architecture"].map(step_map)
cols_prog = ["Step", "Architecture", "Params", "GFLOPs", "FPS", "Mean mIoU", "Mean Dice"]
print("=" * 100)
print("TABLE 1: SYSTEMATIC BOTTLENECK RESOLUTION PROGRESSION STUDY")
print("=" * 100)
print(tabulate(df_prog[cols_prog], headers="keys", tablefmt="github", showindex=False))

# -------------------------------------------------------------
# TABLE 2: MASTER LEADERBOARD ACROSS ALL 15 ARCHITECTURES
# -------------------------------------------------------------
df_ranked = df_master.sort_values(by="Mean mIoU", ascending=False).reset_index(drop=True)
df_ranked.index += 1
cols_rank = ["Architecture", "Category", "Params", "GFLOPs", "FPS", "FloodKaggle mIoU", "AIFloodSense mIoU", "Mean mIoU", "Mean Dice"]
print("\n" + "=" * 100)
print("TABLE 2: MASTER LEADERBOARD ACROSS ALL 15 ARCHITECTURES (Ranked by Mean mIoU)")
print("=" * 100)
print(tabulate(df_ranked[cols_rank], headers="keys", tablefmt="github", showindex=True))

In [ ]:
# 8. Package Results & Download Archive
import shutil, os

os.makedirs("results_bundle", exist_ok=True)
for f in ["hardware_profiling_results.csv", "master_screening_results.csv"]:
    if os.path.exists(f):
        shutil.copy(f, os.path.join("results_bundle", f))

shutil.make_archive("mamba_master_screening_results", "zip", "results_bundle")
print("✓ Results archived to mamba_master_screening_results.zip")

if "google.colab" in str(get_ipython()):
    from google.colab import files
    files.download("mamba_master_screening_results.zip")